# Week 8 — 论文组织、结果复现与项目展示

本 Notebook 是八周课程的正式收尾材料。它不训练新的模型，而是把 Weeks 3--7 的输出整理成一个可复现的科研包：

1. 读取并审计实验 artifacts；
2. 汇总 dataset statistics、model comparison、group CV、top-k screening；
3. 生成 paper-ready tables 和 figure manifest；
4. 生成 mini-paper LaTeX skeleton、presentation outline 和 reproducibility checklist；
5. 用 validation checks 证明结果链路、指标、图表和文件的一致性。

> 核心教学点：科研论文不是把代码跑出来就结束，而是把每个 claim 与可检查的 evidence 对齐。

## 0. Setup

本节只使用标准数据处理和绘图库；所有随机性在前序 Notebook 中已经固定，本周重点是复现实验链路和论文证据链。所有输入来自前几周已经生成的本地 artifacts。

In [1]:
from pathlib import Path
import json
import hashlib
import shutil
import textwrap
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from PIL import Image
    PIL_AVAILABLE = True
except Exception:
    PIL_AVAILABLE = False


def find_course_root(start: Path | None = None) -> Path:
    """Locate the repository root whether Jupyter starts here or in notebooks/."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "environment.yml").exists() and (candidate / "00_overview").is_dir():
            return candidate
    raise FileNotFoundError(
        "Cannot locate the course repository root. Open this notebook from the cloned repository."
    )


PROJECT_ROOT = find_course_root()

def repo_relative(path: Path) -> str:
    """Return a portable repository-relative path for saved manifests."""
    return str(path.relative_to(PROJECT_ROOT))

OUT_DIR = PROJECT_ROOT / '08_week08_paper_reproducibility' / 'outputs'
PAPER_DIR = OUT_DIR / 'paper'
TABLE_DIR = PAPER_DIR / 'tables'
FIG_DIR = PAPER_DIR / 'figures'
PRES_DIR = OUT_DIR / 'presentation'

for d in [OUT_DIR, PAPER_DIR, TABLE_DIR, FIG_DIR, PRES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Output directory:', OUT_DIR)

Output directory: /Users/buxinshe/Git-repo/PINN-Microgrid-Prediction/08_week08_paper_reproducibility/outputs


## 1. Artifact manifest

Week 8 首先要证明：论文使用的所有数据、表格和图都来自可追溯的前序输出。

In [2]:
REQUIRED_ARTIFACTS = {
    # Week 3: base-case scenario generation
    'week3_scenario_table': PROJECT_ROOT / '03_week03_microgrid_scenario_generation/outputs/week3_scenario_table.csv',
    'week3_basecase_features': PROJECT_ROOT / '03_week03_microgrid_scenario_generation/outputs/week3_basecase_ai_features.csv',
    'week3_validation': PROJECT_ROOT / '03_week03_microgrid_scenario_generation/outputs/week3_validation_summary.csv',
    'week3_min_voltage_fig': PROJECT_ROOT / '03_week03_microgrid_scenario_generation/outputs/week3_min_voltage_by_scenario.png',
    'week3_max_vuf_fig': PROJECT_ROOT / '03_week03_microgrid_scenario_generation/outputs/week3_max_vuf_by_scenario.png',
    'week3_max_loading_fig': PROJECT_ROOT / '03_week03_microgrid_scenario_generation/outputs/week3_max_loading_by_scenario.png',

    # Week 4: N-1 labels
    'week4_n1_dataset': PROJECT_ROOT / '04_week04_three_phase_nminus1_labeling/outputs/week4_n1_dataset.csv',
    'week4_ai_ready_dataset': PROJECT_ROOT / '04_week04_three_phase_nminus1_labeling/outputs/week4_ai_ready_dataset.csv',
    'week4_contingency_table': PROJECT_ROOT / '04_week04_three_phase_nminus1_labeling/outputs/week4_contingency_table.csv',
    'week4_label_distribution': PROJECT_ROOT / '04_week04_three_phase_nminus1_labeling/outputs/week4_label_distribution.csv',
    'week4_violation_type_summary': PROJECT_ROOT / '04_week04_three_phase_nminus1_labeling/outputs/week4_violation_type_summary.csv',
    'week4_validation': PROJECT_ROOT / '04_week04_three_phase_nminus1_labeling/outputs/week4_validation_summary.csv',
    'week4_label_distribution_fig': PROJECT_ROOT / '04_week04_three_phase_nminus1_labeling/outputs/week4_label_distribution_by_type.png',
    'week4_severity_fig': PROJECT_ROOT / '04_week04_three_phase_nminus1_labeling/outputs/week4_severity_distribution.png',

    # Week 5: ML baselines
    'week5_holdout_results': PROJECT_ROOT / '05_week05_ml_baselines_screening/outputs/week5_holdout_results.csv',
    'week5_cv_summary': PROJECT_ROOT / '05_week05_ml_baselines_screening/outputs/week5_cv_summary.csv',
    'week5_feature_list': PROJECT_ROOT / '05_week05_ml_baselines_screening/outputs/week5_feature_list.csv',
    'week5_validation': PROJECT_ROOT / '05_week05_ml_baselines_screening/outputs/week5_validation_summary.csv',
    'week5_pr_fig': PROJECT_ROOT / '05_week05_ml_baselines_screening/outputs/week5_precision_recall_curves.png',
    'week5_saved_calls_fig': PROJECT_ROOT / '05_week05_ml_baselines_screening/outputs/week5_saved_calls_vs_missed_violations.png',

    # Week 6: physics-informed MLP
    'week6_holdout_results': PROJECT_ROOT / '06_week06_physics_informed_mlp/outputs/week6_holdout_results.csv',
    'week6_loss_ablation': PROJECT_ROOT / '06_week06_physics_informed_mlp/outputs/week6_loss_ablation_summary.csv',
    'week6_group_cv_summary': PROJECT_ROOT / '06_week06_physics_informed_mlp/outputs/week6_group_cv_summary.csv',
    'week6_component_targets': PROJECT_ROOT / '06_week06_physics_informed_mlp/outputs/week6_component_target_summary.csv',
    'week6_validation': PROJECT_ROOT / '06_week06_physics_informed_mlp/outputs/week6_validation_summary.csv',
    'week6_pr_fig': PROJECT_ROOT / '06_week06_physics_informed_mlp/outputs/week6_precision_recall_curves.png',
    'week6_saved_calls_fig': PROJECT_ROOT / '06_week06_physics_informed_mlp/outputs/week6_saved_calls_vs_missed_violations.png',
    'week6_severity_fig': PROJECT_ROOT / '06_week06_physics_informed_mlp/outputs/week6_true_vs_predicted_severity.png',

    # Week 7: GNN
    'week7_holdout_results': PROJECT_ROOT / '07_week07_phase_aware_gnn_screening/outputs/week7_holdout_results.csv',
    'week7_group_cv_summary': PROJECT_ROOT / '07_week07_phase_aware_gnn_screening/outputs/week7_group_cv_summary.csv',
    'week7_topk': PROJECT_ROOT / '07_week07_phase_aware_gnn_screening/outputs/week7_topk_violation_recall.csv',
    'week7_validation': PROJECT_ROOT / '07_week07_phase_aware_gnn_screening/outputs/week7_validation_summary.csv',
    'week7_pr_fig': PROJECT_ROOT / '07_week07_phase_aware_gnn_screening/outputs/week7_precision_recall_curves.png',
    'week7_saved_calls_fig': PROJECT_ROOT / '07_week07_phase_aware_gnn_screening/outputs/week7_saved_calls_vs_missed_violations.png',
    'week7_topk_fig': PROJECT_ROOT / '07_week07_phase_aware_gnn_screening/outputs/week7_topk_violation_recall.png',
    'week7_group_cv_fig': PROJECT_ROOT / '07_week07_phase_aware_gnn_screening/outputs/week7_group_cv_recall.png',
}

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

artifact_rows = []
for name, path in REQUIRED_ARTIFACTS.items():
    exists = path.exists()
    size = path.stat().st_size if exists else 0
    suffix = path.suffix.lower().lstrip('.')
    digest = sha256_file(path) if exists else None
    artifact_rows.append({
        'artifact': name,
        'path': repo_relative(path),
        'type': suffix,
        'exists': exists,
        'size_bytes': size,
        'sha256': digest,
    })

artifact_manifest = pd.DataFrame(artifact_rows)
artifact_manifest.to_csv(OUT_DIR / 'week8_artifact_manifest.csv', index=False)

assert artifact_manifest['exists'].all(), artifact_manifest.loc[~artifact_manifest['exists']]
assert (artifact_manifest['size_bytes'] > 0).all(), artifact_manifest.loc[artifact_manifest['size_bytes'] <= 0]

artifact_manifest.head(10)

,artifact,path,type,exists,size_bytes,sha256
0,week3_scenario_table,03_week03_microgrid_scenario_generation/output...,csv,True,764,e9e5f7b6e831d7f36940241587e46341f05201e3de9e45...
1,week3_basecase_features,03_week03_microgrid_scenario_generation/output...,csv,True,2408,73743ce60e005be3d6112f26c29b995d45f4e81ffa50b3...
2,week3_validation,03_week03_microgrid_scenario_generation/output...,csv,True,1216,67185fe1ee51e29ff12b0a77998d6cc8d0139dff79d2d3...
3,week3_min_voltage_fig,03_week03_microgrid_scenario_generation/output...,png,True,61397,2757a0b2acd3bf8751691debab8af33992378f084e2250...
4,week3_max_vuf_fig,03_week03_microgrid_scenario_generation/output...,png,True,63851,ee1828da82d4f7fc6db32c7ecb2e0074137c2e46f4ba04...
5,week3_max_loading_fig,03_week03_microgrid_scenario_generation/output...,png,True,63499,9853cc6d6b44104d2401c245610daac9fdb444fa928dc3...
6,week4_n1_dataset,04_week04_three_phase_nminus1_labeling/outputs...,csv,True,22422,0c036f5606c515790c97cfedb0ccea28db57235de5de54...
7,week4_ai_ready_dataset,04_week04_three_phase_nminus1_labeling/outputs...,csv,True,14656,e0bd4ef16fc9cc7899258ad2e01a8ea15b42fbb30ed2cd...
8,week4_contingency_table,04_week04_three_phase_nminus1_labeling/outputs...,csv,True,1003,6fb8c7075d14dba0f90e1e5a684c092b0e199e112898e6...
9,week4_label_distribution,04_week04_three_phase_nminus1_labeling/outputs...,csv,True,164,2499f64973571726d44fb4be6de1f26acd3c829f7139b2...


## 2. Load core datasets

本节读取 Week 3--7 的关键结果表，并做基本形状检查。

In [3]:
def read_csv_artifact(key: str) -> pd.DataFrame:
    path = REQUIRED_ARTIFACTS[key]
    df = pd.read_csv(path)
    assert len(df) > 0, f'{key} is empty'
    return df

scenario_table = read_csv_artifact('week3_scenario_table')
base_features = read_csv_artifact('week3_basecase_features')
n1_dataset = read_csv_artifact('week4_n1_dataset')
ai_ready = read_csv_artifact('week4_ai_ready_dataset')
contingency_table = read_csv_artifact('week4_contingency_table')
label_distribution = read_csv_artifact('week4_label_distribution')
violation_type_summary = read_csv_artifact('week4_violation_type_summary')

week5_holdout = read_csv_artifact('week5_holdout_results')
week5_cv = read_csv_artifact('week5_cv_summary')
week6_holdout = read_csv_artifact('week6_holdout_results')
week6_cv = read_csv_artifact('week6_group_cv_summary')
week6_ablation = read_csv_artifact('week6_loss_ablation')
week7_holdout = read_csv_artifact('week7_holdout_results')
week7_cv = read_csv_artifact('week7_group_cv_summary')
week7_topk = read_csv_artifact('week7_topk')

print('Scenarios:', scenario_table.shape)
print('N-1 dataset:', n1_dataset.shape)
print('AI-ready dataset:', ai_ready.shape)
print('Contingencies:', contingency_table.shape)

Scenarios: (7, 9)
N-1 dataset: (77, 35)
AI-ready dataset: (77, 27)
Contingencies: (11, 6)


## 3. Validation summary audit

各周都已经有 validation summary。Week 8 的第一个 proof 是：前序 validation checks 全部通过。

In [4]:
def summarize_validation(key: str, week: str) -> dict:
    df = read_csv_artifact(key)
    if 'status' in df.columns:
        passed_series = df['status'].astype(str).str.lower().eq('passed')
        check_col = 'test' if 'test' in df.columns else 'check'
    elif 'passed' in df.columns:
        passed_series = df['passed'].astype(bool)
        check_col = 'check'
    else:
        raise ValueError(f'Unknown validation schema for {key}: {df.columns.tolist()}')
    return {
        'week': week,
        'n_checks': int(len(df)),
        'n_passed': int(passed_series.sum()),
        'all_passed': bool(passed_series.all()),
        'failed_checks': ', '.join(df.loc[~passed_series, check_col].astype(str).tolist()),
    }

validation_audit = pd.DataFrame([
    summarize_validation('week3_validation', 'Week 3'),
    summarize_validation('week4_validation', 'Week 4'),
    summarize_validation('week5_validation', 'Week 5'),
    summarize_validation('week6_validation', 'Week 6'),
    summarize_validation('week7_validation', 'Week 7'),
])
validation_audit.to_csv(OUT_DIR / 'week8_validation_audit.csv', index=False)
assert validation_audit['all_passed'].all(), validation_audit
validation_audit

,week,n_checks,n_passed,all_passed,failed_checks
0,Week 3,12,12,True,
1,Week 4,13,13,True,
2,Week 5,12,12,True,
3,Week 6,16,16,True,
4,Week 7,44,44,True,


## 4. Dataset lineage proof

证明 Week 3 的 scenario、Week 4 的 contingency catalog 和 Week 4 的 N-1 dataset 行数一致：

$$
N_{samples} = N_{scenarios}\times N_{contingencies}.
$$

In [5]:
n_scenarios = scenario_table['scenario_id'].nunique()
n_contingencies = contingency_table['contingency_id'].nunique()
expected_samples = n_scenarios * n_contingencies
actual_samples = len(n1_dataset)

assert expected_samples == actual_samples, (expected_samples, actual_samples)
assert n1_dataset[['scenario_id', 'contingency_id']].duplicated().sum() == 0
assert ai_ready[['scenario_id', 'contingency_id']].duplicated().sum() == 0
assert len(ai_ready) == len(n1_dataset)

lineage_summary = pd.DataFrame([{
    'n_scenarios': n_scenarios,
    'n_contingencies': n_contingencies,
    'expected_samples': expected_samples,
    'actual_samples': actual_samples,
    'safe_samples': int((n1_dataset['violation_label'] == False).sum()),
    'unsafe_samples': int((n1_dataset['violation_label'] == True).sum()),
    'power_flow_solved_rows': int((n1_dataset['pf_status'] == 'solved').sum()) if 'pf_status' in n1_dataset.columns else np.nan,
    'service_loss_skipped_rows': int((n1_dataset['pf_status'] == 'service_loss_skip').sum()) if 'pf_status' in n1_dataset.columns else np.nan,
    'no_slack_skipped_rows': int((n1_dataset['pf_status'] == 'no_slack_skip').sum()) if 'pf_status' in n1_dataset.columns else np.nan,
}])
lineage_summary.to_csv(OUT_DIR / 'week8_dataset_lineage_summary.csv', index=False)
lineage_summary

,n_scenarios,n_contingencies,expected_samples,actual_samples,safe_samples,unsafe_samples,power_flow_solved_rows,service_loss_skipped_rows,no_slack_skipped_rows
0,7,11,77,77,30,47,35,35,7


## 5. Label consistency proof

最终 binary label 必须等于各类 violation flags 的 OR，并且 severity 与 safe / unsafe 一致。

In [6]:
violation_cols = [
    'voltage_low_violation', 'voltage_high_violation', 'loading_violation',
    'vuf_violation', 'non_convergence_violation', 'service_loss_violation',
    'critical_service_loss_violation'
]
missing = [c for c in violation_cols if c not in n1_dataset.columns]
assert not missing, missing

flags_or = n1_dataset[violation_cols].astype(bool).any(axis=1)
labels = n1_dataset['violation_label'].astype(bool)
assert (flags_or == labels).all()
assert (n1_dataset.loc[~labels, 'severity_score'].fillna(0) == 0).all()
assert (n1_dataset.loc[labels, 'severity_score'].fillna(0) > 0).all()

violation_summary = pd.DataFrame({
    'violation_type': violation_cols,
    'count': [int(n1_dataset[c].astype(bool).sum()) for c in violation_cols],
})
violation_summary.to_csv(OUT_DIR / 'week8_violation_summary.csv', index=False)
violation_summary

,violation_type,count
0,voltage_low_violation,5
1,voltage_high_violation,0
2,loading_violation,5
3,vuf_violation,0
4,non_convergence_violation,7
5,service_loss_violation,42
6,critical_service_loss_violation,28


## 6. Unified model comparison table

Week 5--7 的结果表列名不完全一致。本节把它们统一成 paper-ready 格式。

In [7]:
def normalize_week5(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({
        'stage': 'Week 5 ML baseline',
        'model': df['model'],
        'recall': df['test_recall'],
        'precision': df['test_precision'],
        'fnr': df['test_fnr'],
        'pf_calls_saved_fraction': df['test_calls_saved_fraction'],
        'missed_violations': df['test_missed_violations'],
        'auprc': df['test_auprc'],
        'auroc': df['test_roc_auc'],
        'selected_threshold': df['selected_threshold'],
    })
    return out

def normalize_week6(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({
        'stage': 'Week 6 PI-MLP',
        'model': df['model'],
        'recall': df['recall'],
        'precision': df['precision'],
        'fnr': df['fnr'],
        'pf_calls_saved_fraction': df['pf_calls_saved_fraction'],
        'missed_violations': df['missed_violations'],
        'auprc': df['auprc'],
        'auroc': df['auroc'],
        'selected_threshold': df['selected_threshold'],
    })
    return out

def normalize_week7(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({
        'stage': 'Week 7 GNN',
        'model': df['model'],
        'recall': df['recall'],
        'precision': df['precision'],
        'fnr': df['FNR'],
        'pf_calls_saved_fraction': df['pf_calls_saved_fraction'],
        'missed_violations': df['missed_violations'],
        'auprc': df['test_AUPRC'],
        'auroc': df['test_AUROC'],
        'selected_threshold': df['selected_threshold'],
    })
    return out

model_comparison = pd.concat([
    normalize_week5(week5_holdout),
    normalize_week6(week6_holdout),
    normalize_week7(week7_holdout),
], ignore_index=True)

metric_cols = ['recall','precision','fnr','pf_calls_saved_fraction','missed_violations','auprc','auroc','selected_threshold']
assert np.isfinite(model_comparison[metric_cols].to_numpy(dtype=float)).all()
for c in ['recall','precision','fnr','pf_calls_saved_fraction','auprc','auroc']:
    assert ((model_comparison[c] >= 0) & (model_comparison[c] <= 1)).all(), c
assert (model_comparison['missed_violations'] >= 0).all()

model_comparison = model_comparison.sort_values(
    by=['missed_violations','recall','pf_calls_saved_fraction','precision'],
    ascending=[True, False, False, False]
).reset_index(drop=True)

model_comparison.to_csv(OUT_DIR / 'week8_model_comparison_table.csv', index=False)
model_comparison

,stage,model,recall,precision,fnr,pf_calls_saved_fraction,missed_violations,auprc,auroc,selected_threshold
0,Week 5 ML baseline,GradientBoosting,1.000000,1.000000,0.000000,0.40,0,1.000000,1.000000,0.300919
1,Week 6 PI-MLP,PI-MLP full,1.000000,1.000000,0.000000,0.40,0,1.000000,1.000000,0.005000
2,Week 7 GNN,FlattenedMLP,1.000000,1.000000,0.000000,0.40,0,1.000000,1.000000,0.994366
3,Week 5 ML baseline,EngineeringRule,1.000000,0.857143,0.000000,0.30,0,0.910053,0.906250,0.650000
4,Week 6 PI-MLP,BaselineMLP,1.000000,0.857143,0.000000,0.30,0,1.000000,1.000000,0.010000
5,Week 5 ML baseline,AllUnsafe,1.000000,0.600000,0.000000,0.00,0,0.600000,0.500000,0.000000
6,Week 5 ML baseline,MLP,1.000000,0.600000,0.000000,0.00,0,0.533396,0.239583,0.000000
7,Week 5 ML baseline,LogisticRegression,0.833333,1.000000,0.166667,0.50,2,1.000000,1.000000,0.734750
8,Week 5 ML baseline,RandomForest,0.833333,0.909091,0.166667,0.45,2,0.955982,0.937500,0.677342
9,Week 7 GNN,PhaseAwareGNN,0.583333,0.875000,0.416667,0.60,5,0.912543,0.875000,0.926189


## 7. Group-CV synthesis

随机划分容易高估性能。最终报告应同时展示 scenario-group 和 contingency-group 的泛化结果。

In [8]:
def norm_week5_cv(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        'stage': 'Week 5 ML baseline',
        'model': df['model'],
        'cv_type': df['cv_protocol'],
        'folds': df['folds'],
        'mean_recall': df['recall_mean'],
        'mean_fnr': df['fnr_mean'],
        'mean_pf_calls_saved': df['calls_saved_mean'],
        'mean_auprc': df['auprc_mean'],
        'missed_violations_sum': df['missed_violations_sum'],
    })

def norm_week6_cv(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        'stage': 'Week 6 PI-MLP',
        'model': 'PI-MLP full',
        'cv_type': df['cv_type'].replace({'scenario_id':'scenario_group','contingency_id':'contingency_group'}),
        'folds': df['folds'],
        'mean_recall': df['mean_recall'],
        'mean_fnr': df['mean_fnr'],
        'mean_pf_calls_saved': df['mean_pf_calls_saved'],
        'mean_auprc': df['mean_auprc'],
        'missed_violations_sum': np.nan,
    })

def norm_week7_cv(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        'stage': 'Week 7 GNN',
        'model': 'PhaseAwareGNN',
        'cv_type': df['cv_type'],
        'folds': df['n_folds'],
        'mean_recall': df['mean_recall'],
        'mean_fnr': df['mean_FNR'],
        'mean_pf_calls_saved': df['mean_pf_calls_saved'],
        'mean_auprc': np.nan,
        'missed_violations_sum': df['total_missed_violations'],
    })

group_cv_synthesis = pd.concat([
    norm_week5_cv(week5_cv),
    norm_week6_cv(week6_cv),
    norm_week7_cv(week7_cv),
], ignore_index=True)

for c in ['mean_recall','mean_fnr','mean_pf_calls_saved']:
    assert group_cv_synthesis[c].dropna().between(0,1).all(), c
assert set(group_cv_synthesis['cv_type'].astype(str).str.replace('scenario_id','scenario_group').str.replace('contingency_id','contingency_group')).issuperset({'scenario_group','contingency_group'})

group_cv_synthesis.to_csv(OUT_DIR / 'week8_group_cv_synthesis.csv', index=False)
group_cv_synthesis.head(12)

,stage,model,cv_type,folds,mean_recall,mean_fnr,mean_pf_calls_saved,mean_auprc,missed_violations_sum
0,Week 5 ML baseline,EngineeringRule,scenario_group,3,1.000000,0.000000,0.323232,0.911789,0.0
1,Week 5 ML baseline,AllUnsafe,scenario_group,3,1.000000,0.000000,0.000000,0.595960,0.0
2,Week 5 ML baseline,LogisticRegression,scenario_group,3,0.927536,0.072464,0.439394,0.990367,5.0
3,Week 5 ML baseline,RandomForest,scenario_group,3,0.898551,0.101449,0.474747,0.989687,7.0
4,Week 5 ML baseline,EngineeringRule,contingency_group,3,1.000000,0.000000,0.309524,0.970835,0.0
5,Week 5 ML baseline,AllUnsafe,contingency_group,3,1.000000,0.000000,0.000000,0.619048,0.0
6,Week 5 ML baseline,RandomForest,contingency_group,3,0.400000,0.600000,0.690476,0.860834,28.0
7,Week 5 ML baseline,LogisticRegression,contingency_group,3,0.254167,0.745833,0.785714,0.909407,35.0
8,Week 6 PI-MLP,PI-MLP full,contingency_group,3,0.759722,0.240278,0.412698,0.877075,NaN
9,Week 6 PI-MLP,PI-MLP full,scenario_group,3,0.927536,0.072464,0.207071,0.941912,NaN


## 8. Top-k screening proof

Top-k contingency screening 是部署故事的重要部分。这里检查 top-k 表的单调性和数值范围。

In [9]:
assert {'k','pf_call_fraction','mean_violation_recall_at_k','min_violation_recall_at_k','severe_hit_rate_at_k'}.issubset(week7_topk.columns)
assert week7_topk['k'].is_monotonic_increasing
for c in ['pf_call_fraction','mean_violation_recall_at_k','min_violation_recall_at_k','severe_hit_rate_at_k']:
    assert week7_topk[c].between(0,1).all(), c
assert week7_topk['mean_violation_recall_at_k'].is_monotonic_increasing
assert week7_topk['pf_call_fraction'].is_monotonic_increasing

week7_topk.to_csv(OUT_DIR / 'week8_topk_table.csv', index=False)
week7_topk

,k,pf_call_fraction,mean_violation_recall_at_k,min_violation_recall_at_k,severe_hit_rate_at_k
0,1,0.090909,0.155844,0.090909,1.0
1,2,0.181818,0.311688,0.181818,1.0
2,3,0.272727,0.467532,0.272727,1.0
3,4,0.363636,0.575758,0.363636,1.0
4,5,0.454545,0.636364,0.454545,1.0
5,6,0.545455,0.792208,0.545455,1.0
6,7,0.636364,0.948052,0.636364,1.0
7,8,0.727273,0.961039,0.727273,1.0
8,9,0.818182,0.974026,0.818182,1.0
9,10,0.909091,0.987013,0.909091,1.0


## 9. No-leakage feature audit

最终论文应明确说明输入特征只来自 base-case 和 contingency metadata。这里复查 Week 5/7 的特征列表。

In [10]:
PROHIBITED_PATTERNS = [
    'post', 'violation', 'severity_score', 'severity', 'converged', 'pf_status',
    'non_convergence', 'service_loss', 'critical_service_loss', 'unserved',
    'min_vm_pu', 'max_vm_pu', 'max_vuf_percent', 'max_line_loading_percent'
]

feature_audits = []
for key in ['week5_feature_list']:
    df = read_csv_artifact(key)
    feature_col = 'feature' if 'feature' in df.columns else df.columns[0]
    features = df[feature_col].astype(str).tolist()
    bad = [f for f in features if any(p in f.lower() and not f.lower().startswith('base_') for p in PROHIBITED_PATTERNS)]
    feature_audits.append({'source': key, 'n_features': len(features), 'n_suspicious': len(bad), 'suspicious_features': '; '.join(bad[:20])})

# Week 7 has node / edge / global feature lists.
for key, path in {
    'week7_node_feature_list': PROJECT_ROOT / '07_week07_phase_aware_gnn_screening/outputs/week7_node_feature_list.csv',
    'week7_edge_feature_list': PROJECT_ROOT / '07_week07_phase_aware_gnn_screening/outputs/week7_edge_feature_list.csv',
    'week7_global_feature_list': PROJECT_ROOT / '07_week07_phase_aware_gnn_screening/outputs/week7_global_feature_list.csv',
}.items():
    df = pd.read_csv(path)
    feature_col = 'feature' if 'feature' in df.columns else df.columns[0]
    features = df[feature_col].astype(str).tolist()
    bad = [f for f in features if any(p in f.lower() and not f.lower().startswith('base_') for p in PROHIBITED_PATTERNS)]
    feature_audits.append({'source': key, 'n_features': len(features), 'n_suspicious': len(bad), 'suspicious_features': '; '.join(bad[:20])})

feature_audit = pd.DataFrame(feature_audits)
feature_audit.to_csv(OUT_DIR / 'week8_feature_leakage_audit.csv', index=False)
assert (feature_audit['n_suspicious'] == 0).all(), feature_audit
feature_audit

,source,n_features,n_suspicious,suspicious_features
0,week5_feature_list,32,0,
1,week7_node_feature_list,26,0,
2,week7_edge_feature_list,6,0,
3,week7_global_feature_list,16,0,


## 10. Paper-ready tables

生成论文中最常用的四张表：dataset statistics、model comparison、group CV、loss ablation。

In [11]:
def latex_escape(s):
    if pd.isna(s):
        return ''
    return str(s).replace('_', r'\_').replace('%', r'\%').replace('&', r'\&')

def save_latex_table(df: pd.DataFrame, path: Path, caption: str, label: str, float_format='%.3f'):
    """Write a compilable wide IEEE table with escaped headers and text values."""
    safe = df.copy()
    safe.columns = [latex_escape(col) for col in safe.columns]
    for col in safe.select_dtypes(include='object').columns:
        safe[col] = safe[col].map(latex_escape)
    tabular = safe.to_latex(index=False, escape=False, float_format=float_format)
    latex = (
        "\\begin{table*}[t]\n"
        "\\centering\n"
        f"\\caption{{{caption}}}\n"
        f"\\label{{{label}}}\n"
        "\\resizebox{\\textwidth}{!}{%\n"
        + tabular
        + "}\n\\end{table*}\n"
    )
    path.write_text(latex, encoding='utf-8')
    assert path.exists() and path.stat().st_size > 0
    return latex

# Table I: dataset statistics
dataset_stats = lineage_summary.copy()
dataset_stats.insert(0, 'dataset', 'Three-phase microgrid N-1 dataset')
dataset_stats.to_csv(TABLE_DIR / 'table1_dataset_statistics.csv', index=False)
save_latex_table(dataset_stats, TABLE_DIR / 'table1_dataset_statistics.tex',
                 'Dataset statistics for the teaching microgrid N-1 benchmark.', 'tab:dataset')

# Table II: model comparison
table2 = model_comparison[['stage','model','recall','precision','fnr','pf_calls_saved_fraction','missed_violations','auprc','auroc']].copy()
table2.to_csv(TABLE_DIR / 'table2_model_comparison.csv', index=False)
save_latex_table(table2, TABLE_DIR / 'table2_model_comparison.tex',
                 'Holdout comparison of security screening models.', 'tab:model_comparison')

# Table III: group CV
table3 = group_cv_synthesis[['stage','model','cv_type','folds','mean_recall','mean_fnr','mean_pf_calls_saved','mean_auprc','missed_violations_sum']].copy()
table3.to_csv(TABLE_DIR / 'table3_group_cv.csv', index=False)
save_latex_table(table3, TABLE_DIR / 'table3_group_cv.tex',
                 'Group-based cross-validation for unseen scenarios and contingencies.', 'tab:group_cv')

# Table IV: loss ablation
ablation = week6_ablation.copy()
ablation.to_csv(TABLE_DIR / 'table4_loss_ablation.csv', index=False)
save_latex_table(ablation, TABLE_DIR / 'table4_loss_ablation.tex',
                 'Loss ablation for physics-informed MLP training.', 'tab:ablation')

print('Saved paper tables to', TABLE_DIR)

Saved paper tables to /Users/buxinshe/Git-repo/PINN-Microgrid-Prediction/08_week08_paper_reproducibility/outputs/paper/tables


## 11. Figure manifest and audit

把最终论文建议使用的图复制到 `paper/figures/`，并记录尺寸、文件大小和来源。

In [12]:
FIGURE_SOURCES = {
    'fig1_week4_label_distribution_by_type.png': REQUIRED_ARTIFACTS['week4_label_distribution_fig'],
    'fig2_week4_severity_distribution.png': REQUIRED_ARTIFACTS['week4_severity_fig'],
    'fig3_week6_precision_recall_curves.png': REQUIRED_ARTIFACTS['week6_pr_fig'],
    'fig4_week6_saved_calls_vs_missed_violations.png': REQUIRED_ARTIFACTS['week6_saved_calls_fig'],
    'fig5_week7_topk_violation_recall.png': REQUIRED_ARTIFACTS['week7_topk_fig'],
    'fig6_week7_group_cv_recall.png': REQUIRED_ARTIFACTS['week7_group_cv_fig'],
    'fig7_week6_true_vs_predicted_severity.png': REQUIRED_ARTIFACTS['week6_severity_fig'],
}

fig_rows = []
for out_name, src in FIGURE_SOURCES.items():
    dst = FIG_DIR / out_name
    shutil.copy2(src, dst)
    row = {
        'figure_file': out_name,
        'source_path': str(src),
        'copied_path': str(dst),
        'size_bytes': dst.stat().st_size,
        'sha256': sha256_file(dst),
    }
    if PIL_AVAILABLE:
        with Image.open(dst) as im:
            row['width_px'], row['height_px'] = im.size
            row['mode'] = im.mode
    fig_rows.append(row)

figure_manifest = pd.DataFrame(fig_rows)
figure_manifest.to_csv(OUT_DIR / 'week8_figure_manifest.csv', index=False)
assert (figure_manifest['size_bytes'] > 0).all()
if PIL_AVAILABLE:
    assert (figure_manifest['width_px'] > 100).all()
    assert (figure_manifest['height_px'] > 100).all()
figure_manifest

,figure_file,source_path,copied_path,size_bytes,sha256,width_px,height_px,mode
0,fig1_week4_label_distribution_by_type.png,/Users/buxinshe/Git-repo/PINN-Microgrid-Predic...,/Users/buxinshe/Git-repo/PINN-Microgrid-Predic...,46976,97c25d9645bc0a15ddf196ff2bd1f900856a9d3bb54863...,1440,810,RGBA
1,fig2_week4_severity_distribution.png,/Users/buxinshe/Git-repo/PINN-Microgrid-Predic...,/Users/buxinshe/Git-repo/PINN-Microgrid-Predic...,32316,2ccb36982791bc1adbe5687b2ff973b279864246bf3dc5...,1440,810,RGBA
2,fig3_week6_precision_recall_curves.png,/Users/buxinshe/Git-repo/PINN-Microgrid-Predic...,/Users/buxinshe/Git-repo/PINN-Microgrid-Predic...,27489,904f78d109f6f6e14095a34b8a58f76e2ba8940d010eb7...,1120,640,RGBA
3,fig4_week6_saved_calls_vs_missed_violations.png,/Users/buxinshe/Git-repo/PINN-Microgrid-Predic...,/Users/buxinshe/Git-repo/PINN-Microgrid-Predic...,33670,5a10ef8dcccdc1fdf78c2f05c7d5b4ad39b419098ca388...,1120,640,RGBA
4,fig5_week7_topk_violation_recall.png,/Users/buxinshe/Git-repo/PINN-Microgrid-Predic...,/Users/buxinshe/Git-repo/PINN-Microgrid-Predic...,70110,221e1023f7fb556d7c063e790976b30dbe1c2a0683a022...,1080,720,RGBA
5,fig6_week7_group_cv_recall.png,/Users/buxinshe/Git-repo/PINN-Microgrid-Predic...,/Users/buxinshe/Git-repo/PINN-Microgrid-Predic...,29474,e41d6d6f1110dff9a61b02937e8c67218628977a58fc16...,1080,720,RGBA
6,fig7_week6_true_vs_predicted_severity.png,/Users/buxinshe/Git-repo/PINN-Microgrid-Predic...,/Users/buxinshe/Git-repo/PINN-Microgrid-Predic...,23643,23b85864b7673e59282bad28b1c727322e518e3e53ba43...,960,640,RGBA


## 12. Claim–evidence table

论文中的每一个 claim 都应该能追溯到表格、图或 validation proof。

In [13]:
best_zero_miss = model_comparison[model_comparison['missed_violations'] == 0].copy()
best_model = best_zero_miss.sort_values(['pf_calls_saved_fraction','precision'], ascending=False).iloc[0]

claim_evidence = pd.DataFrame([
    {
        'claim_id': 'C1',
        'claim': 'The workflow creates a complete three-phase N-1 dataset from scenario and contingency catalogs.',
        'evidence': 'week8_dataset_lineage_summary.csv; table1_dataset_statistics.tex',
        'status': 'supported',
    },
    {
        'claim_id': 'C2',
        'claim': 'The binary violation label is consistent with voltage, loading, VUF, service-loss, and non-convergence flags.',
        'evidence': 'week8_violation_summary.csv; label consistency proof in Notebook 08',
        'status': 'supported',
    },
    {
        'claim_id': 'C3',
        'claim': f"The best zero-missed-violation holdout model is {best_model['model']} with PF calls saved fraction {best_model['pf_calls_saved_fraction']:.3f}.",
        'evidence': 'week8_model_comparison_table.csv; table2_model_comparison.tex',
        'status': 'supported',
    },
    {
        'claim_id': 'C4',
        'claim': 'Group-based CV provides a stricter test than random holdout because scenarios and contingencies are separated by group.',
        'evidence': 'week8_group_cv_synthesis.csv; table3_group_cv.tex',
        'status': 'supported',
    },
    {
        'claim_id': 'C5',
        'claim': 'Top-k screening provides an operator-oriented way to reduce the contingency list sent to three-phase power flow.',
        'evidence': 'week8_topk_table.csv; fig5_week7_topk_violation_recall.png',
        'status': 'supported',
    },
    {
        'claim_id': 'L1',
        'claim': 'The current benchmark is a small teaching feeder, so results should not be claimed as broad deployment evidence.',
        'evidence': 'dataset statistics and Week 7 discussion; future work section',
        'status': 'limitation',
    },
])
claim_evidence.to_csv(OUT_DIR / 'week8_claim_evidence_table.csv', index=False)
claim_evidence

,claim_id,claim,evidence,status
0,C1,The workflow creates a complete three-phase N-...,week8_dataset_lineage_summary.csv; table1_data...,supported
1,C2,The binary violation label is consistent with ...,week8_violation_summary.csv; label consistency...,supported
2,C3,The best zero-missed-violation holdout model i...,week8_model_comparison_table.csv; table2_model...,supported
3,C4,Group-based CV provides a stricter test than r...,week8_group_cv_synthesis.csv; table3_group_cv.tex,supported
4,C5,Top-k screening provides an operator-oriented ...,week8_topk_table.csv; fig5_week7_topk_violatio...,supported
5,L1,The current benchmark is a small teaching feed...,dataset statistics and Week 7 discussion; futu...,limitation


## 13. Generate mini-paper skeleton

这里生成一个可直接继续写作的 LaTeX skeleton。正式版可在 Week 8 后继续补充正文、引用和图。

In [14]:
paper_tex = r"""
\documentclass[conference]{IEEEtran}
\usepackage{amsmath,amssymb}
\usepackage{graphicx}
\usepackage{booktabs}
\usepackage{multirow}
\usepackage{url}

\title{Physics-informed AI for Fast Three-phase N-1 Security Screening in Unbalanced Microgrids}
\author{Student Name, Advisor Name}

\begin{document}
\maketitle

\begin{abstract}
This mini-paper studies fast static N-1 security screening for unbalanced microgrids. A pandapower-based three-phase power-flow pipeline is used to generate post-contingency labels under voltage, loading, voltage-unbalance, service-loss, and convergence criteria. Several screening models are compared, including traditional ML baselines, a physics-informed multi-task MLP, and a phase-aware GNN. The goal is not to replace three-phase power flow, but to prioritize high-risk contingencies while maintaining high recall.
\end{abstract}

\section{Introduction}
Unbalanced microgrids and low-voltage feeders can experience phase-specific voltage, current, and unbalance violations under component outages. Exhaustive three-phase N-1 scans are accurate but expensive when many operating scenarios and contingencies must be assessed. This work evaluates AI-based pre-screening models for fast high-recall contingency prioritization.

\textbf{Contributions:}
\begin{itemize}
    \item A reproducible three-phase N-1 labeling workflow for microgrid security screening.
    \item A phase-aware violation definition including voltage, loading, VUF, service-loss, and non-convergence labels.
    \item A comparison of engineering rules, ML baselines, PI-MLP, and phase-aware GNN models under high-recall screening metrics.
    \item A validation suite for dataset lineage, no-leakage feature design, metric consistency, and graph invariance checks.
\end{itemize}

\section{Three-phase Microgrid Security Formulation}
For each base-case scenario $x_t$ and contingency $c\in\mathcal{C}$, the target is a binary label $y_{t,c}$ and severity score $s_{t,c}$. The post-contingency quantities include phase voltages $V_{i,\phi}^{c}$, line currents $I_{\ell,\phi}^{c}$, and voltage unbalance factors.

\subsection{Violation Labels}
The final label is the logical OR of low-voltage, high-voltage, loading, VUF, service-loss, critical service-loss, and non-convergence flags.

\section{Dataset Generation}
The dataset is generated using a microgrid-like teaching feeder. Scenario generation controls load level, phase imbalance, PV penetration, and BESS charging/discharging setpoints. Each scenario is combined with a contingency catalog including line outages, PV trips, BESS trip, and PCC outage.

\input{tables/table1_dataset_statistics.tex}

\section{Screening Models}
\subsection{Traditional ML Baselines}
Engineering rules, logistic regression, random forest, gradient boosting, and MLP baselines are trained using no-leakage base-case features.

\subsection{Physics-informed MLP}
The PI-MLP predicts a violation probability, severity score, and component margins. The loss combines classification, severity, component-margin, component-severity consistency, probability-severity consistency, and ranking terms.

\subsection{Phase-aware GNN}
The GNN uses bus-level phase-channel node features, line edge features, and contingency masks to produce a graph-level risk score.

\section{Experiments}
\input{tables/table2_model_comparison.tex}
\input{tables/table3_group_cv.tex}
\input{tables/table4_loss_ablation.tex}

\begin{figure}[t]
    \centering
    \includegraphics[width=0.95\linewidth]{figures/fig3_week6_precision_recall_curves.png}
    \caption{Precision-recall curves for screening models.}
    \label{fig:pr}
\end{figure}

\begin{figure}[t]
    \centering
    \includegraphics[width=0.95\linewidth]{figures/fig4_week6_saved_calls_vs_missed_violations.png}
    \caption{Tradeoff between saved three-phase power-flow calls and missed violations.}
    \label{fig:saved_calls}
\end{figure}

\begin{figure}[t]
    \centering
    \includegraphics[width=0.95\linewidth]{figures/fig5_week7_topk_violation_recall.png}
    \caption{Top-$k$ violation recall for contingency prioritization.}
    \label{fig:topk}
\end{figure}

\section{Discussion and Limitations}
The current benchmark is intentionally small and teaching-oriented. It supports methodological development and validation logic, but broader claims require additional feeders, stochastic scenarios, islanded-mode modeling, grid-forming inverter models, and real microgrid data.

\section{Conclusion}
This project demonstrates a reproducible workflow for three-phase microgrid N-1 security screening with AI. The main lesson is that high-recall screening, validation discipline, and no-leakage feature design are as important as model architecture.

\end{document}
"""

paper_path = PAPER_DIR / 'main.tex'
paper_path.write_text(paper_tex.strip() + '\n', encoding='utf-8')
assert paper_path.exists() and paper_path.stat().st_size > 0
print('Wrote', paper_path)

Wrote /Users/buxinshe/Git-repo/PINN-Microgrid-Prediction/08_week08_paper_reproducibility/outputs/paper/main.tex


## 14. Generate presentation outline

生成最终答辩 PPT 的 10--15 页结构。后续可以把它转换成 Beamer 或 PowerPoint。

In [15]:
presentation_outline = r"""
# Final Presentation Outline

## Slide 1. Title
Physics-informed AI for Fast Three-phase N-1 Security Screening in Unbalanced Microgrids

## Slide 2. Problem motivation
Unbalanced microgrids require phase-aware static security checks; exhaustive three-phase N-1 scans are accurate but expensive.

## Slide 3. Research question
Can AI pre-screening maintain high recall while reducing the number of post-contingency `runpp_3ph()` calls?

## Slide 4. Dataset pipeline
Scenario generation -> N-1 contingency catalog -> three-phase power flow -> violation labels.

## Slide 5. Violation definition
Voltage, loading, VUF, service-loss, critical service-loss, and non-convergence labels.

## Slide 6. Screening formulation
Input: base-case features + contingency metadata. Output: risk probability / severity / top-k ranking.

## Slide 7. Baselines
Engineering rule, Logistic Regression, Random Forest, Gradient Boosting, MLP.

## Slide 8. Physics-informed MLP
Multi-head prediction with component margins and consistency losses.

## Slide 9. Phase-aware GNN
Bus-level phase-channel graph with line outage masks and graph-level risk readout.

## Slide 10. Holdout results
Show Table II and explain recall, FNR, PF calls saved, missed violations.

## Slide 11. Screening tradeoff
Show saved PF calls vs missed violations.

## Slide 12. OOD-style evaluation
Scenario-group and contingency-group CV.

## Slide 13. Top-k contingency prioritization
Show top-k violation recall.

## Slide 14. Validation suite
Dataset lineage, no-leakage, metric proof, graph invariance, reproducibility.

## Slide 15. Limitations and future work
More feeders, islanding, grid-forming inverter, phase-level GNN, conformal screening, real data.
"""

outline_path = PRES_DIR / 'final_presentation_outline.md'
outline_path.write_text(presentation_outline.strip() + '\n', encoding='utf-8')
assert outline_path.exists() and outline_path.stat().st_size > 0
print('Wrote', outline_path)

Wrote /Users/buxinshe/Git-repo/PINN-Microgrid-Prediction/08_week08_paper_reproducibility/outputs/presentation/final_presentation_outline.md


## 15. Reproducibility checklist

生成论文包 checklist，明确学生最终提交时需要检查的项目。

In [16]:
checklist_items = [
    ('data_lineage', 'Scenario count × contingency count equals N-1 dataset rows.', True),
    ('label_consistency', 'Final violation label equals OR of all component violation flags.', True),
    ('validation_summaries', 'Week 3--7 validation summaries all pass.', bool(validation_audit['all_passed'].all())),
    ('no_leakage', 'Feature lists do not include post-contingency labels or metrics.', True),
    ('threshold_protocol', 'Threshold is selected on validation set, not test set.', True),
    ('group_cv', 'Scenario-group and contingency-group CV results are included.', True),
    ('topk', 'Top-k screening table is finite and monotonic.', True),
    ('paper_tables', 'LaTeX tables are generated under paper/tables.', True),
    ('paper_figures', 'Paper figures are copied under paper/figures with manifest.', True),
    ('claim_evidence', 'Each main claim has a linked evidence artifact.', True),
    ('limitations', 'Limitations are explicitly stated.', True),
]
repro_checklist = pd.DataFrame(checklist_items, columns=['item','description','passed'])
repro_checklist.to_csv(OUT_DIR / 'week8_reproducibility_checklist.csv', index=False)
assert repro_checklist['passed'].all()
repro_checklist

,item,description,passed
0,data_lineage,Scenario count × contingency count equals N-1 ...,True
1,label_consistency,Final violation label equals OR of all compone...,True
2,validation_summaries,Week 3--7 validation summaries all pass.,True
3,no_leakage,Feature lists do not include post-contingency ...,True
4,threshold_protocol,"Threshold is selected on validation set, not t...",True
5,group_cv,Scenario-group and contingency-group CV result...,True
6,topk,Top-k screening table is finite and monotonic.,True
7,paper_tables,LaTeX tables are generated under paper/tables.,True
8,paper_figures,Paper figures are copied under paper/figures w...,True
9,claim_evidence,Each main claim has a linked evidence artifact.,True


## 16. Mini report: key numbers and suggested claims

本节将关键数字组合成一段可放入报告的结果描述。

In [17]:
best_zero_miss = model_comparison[model_comparison['missed_violations'] == 0].copy()
if len(best_zero_miss) > 0:
    chosen = best_zero_miss.sort_values(['pf_calls_saved_fraction','precision','auprc'], ascending=False).iloc[0]
else:
    chosen = model_comparison.sort_values(['recall','pf_calls_saved_fraction'], ascending=False).iloc[0]

summary_text = f"""
The final teaching dataset contains {int(lineage_summary.loc[0, 'actual_samples'])} scenario-contingency samples from {n_scenarios} operating scenarios and {n_contingencies} contingencies. There are {int(lineage_summary.loc[0, 'unsafe_samples'])} unsafe samples and {int(lineage_summary.loc[0, 'safe_samples'])} safe samples. In the holdout comparison, the strongest zero-missed-violation model is {chosen['model']} from {chosen['stage']}, achieving recall={chosen['recall']:.3f}, precision={chosen['precision']:.3f}, FNR={chosen['fnr']:.3f}, and PF calls saved fraction={chosen['pf_calls_saved_fraction']:.3f}. Group-based cross-validation and top-k contingency prioritization are included to evaluate scenario and contingency generalization beyond random holdout splits.
""".strip()

(OUT_DIR / 'week8_key_result_summary.txt').write_text(summary_text + '\n', encoding='utf-8')
print(summary_text)

The final teaching dataset contains 77 scenario-contingency samples from 7 operating scenarios and 11 contingencies. There are 47 unsafe samples and 30 safe samples. In the holdout comparison, the strongest zero-missed-violation model is GradientBoosting from Week 5 ML baseline, achieving recall=1.000, precision=1.000, FNR=0.000, and PF calls saved fraction=0.400. Group-based cross-validation and top-k contingency prioritization are included to evaluate scenario and contingency generalization beyond random holdout splits.


## 17. Final validation summary

Week 8 最后一项检查：所有本周生成的核心输出存在、非空，且关键 proof 已通过。

In [18]:
week8_generated = {
    'artifact_manifest': OUT_DIR / 'week8_artifact_manifest.csv',
    'validation_audit': OUT_DIR / 'week8_validation_audit.csv',
    'dataset_lineage': OUT_DIR / 'week8_dataset_lineage_summary.csv',
    'violation_summary': OUT_DIR / 'week8_violation_summary.csv',
    'model_comparison': OUT_DIR / 'week8_model_comparison_table.csv',
    'group_cv_synthesis': OUT_DIR / 'week8_group_cv_synthesis.csv',
    'topk_table': OUT_DIR / 'week8_topk_table.csv',
    'feature_leakage_audit': OUT_DIR / 'week8_feature_leakage_audit.csv',
    'figure_manifest': OUT_DIR / 'week8_figure_manifest.csv',
    'claim_evidence': OUT_DIR / 'week8_claim_evidence_table.csv',
    'reproducibility_checklist': OUT_DIR / 'week8_reproducibility_checklist.csv',
    'mini_paper_tex': PAPER_DIR / 'main.tex',
    'presentation_outline': PRES_DIR / 'final_presentation_outline.md',
    'key_result_summary': OUT_DIR / 'week8_key_result_summary.txt',
}

final_checks = []
for name, path in week8_generated.items():
    final_checks.append({
        'check': f'generated_{name}',
        'passed': path.exists() and path.stat().st_size > 0,
        'detail': repo_relative(path),
    })

# Add explicit logical proof checks.
final_checks.extend([
    {'check': 'required_artifacts_exist', 'passed': bool(artifact_manifest['exists'].all()), 'detail': f"n={len(artifact_manifest)}"},
    {'check': 'all_prior_validations_passed', 'passed': bool(validation_audit['all_passed'].all()), 'detail': f"weeks={len(validation_audit)}"},
    {'check': 'sample_count_consistency', 'passed': bool(expected_samples == actual_samples), 'detail': f"{expected_samples}={actual_samples}"},
    {'check': 'label_or_consistency', 'passed': bool((flags_or == labels).all()), 'detail': f"rows={len(labels)}"},
    {'check': 'model_metrics_bounded', 'passed': bool(model_comparison[['recall','precision','fnr','pf_calls_saved_fraction','auprc','auroc']].apply(lambda s: s.between(0,1).all()).all()), 'detail': f"models={len(model_comparison)}"},
    {'check': 'topk_monotonic', 'passed': bool(week7_topk['mean_violation_recall_at_k'].is_monotonic_increasing and week7_topk['pf_call_fraction'].is_monotonic_increasing), 'detail': f"rows={len(week7_topk)}"},
    {'check': 'no_feature_leakage_audit', 'passed': bool((feature_audit['n_suspicious'] == 0).all()), 'detail': f"sources={len(feature_audit)}"},
    {'check': 'figure_audit', 'passed': bool((figure_manifest['size_bytes'] > 0).all()), 'detail': f"figures={len(figure_manifest)}"},
])

week8_validation_summary = pd.DataFrame(final_checks)
week8_validation_summary.to_csv(OUT_DIR / 'week8_validation_summary.csv', index=False)
assert week8_validation_summary['passed'].all(), week8_validation_summary.loc[~week8_validation_summary['passed']]

print(f"Week 8 validation checks: {week8_validation_summary['passed'].sum()} / {len(week8_validation_summary)} passed")
week8_validation_summary

Week 8 validation checks: 22 / 22 passed


,check,passed,detail
0,generated_artifact_manifest,True,08_week08_paper_reproducibility/outputs/week8_...
1,generated_validation_audit,True,08_week08_paper_reproducibility/outputs/week8_...
2,generated_dataset_lineage,True,08_week08_paper_reproducibility/outputs/week8_...
3,generated_violation_summary,True,08_week08_paper_reproducibility/outputs/week8_...
4,generated_model_comparison,True,08_week08_paper_reproducibility/outputs/week8_...
5,generated_group_cv_synthesis,True,08_week08_paper_reproducibility/outputs/week8_...
6,generated_topk_table,True,08_week08_paper_reproducibility/outputs/week8_...
7,generated_feature_leakage_audit,True,08_week08_paper_reproducibility/outputs/week8_...
8,generated_figure_manifest,True,08_week08_paper_reproducibility/outputs/week8_...
9,generated_claim_evidence,True,08_week08_paper_reproducibility/outputs/week8_...


## 18. Cross-validation evidence synthesis

本节不是重新训练模型，而是把 Week 5--7 已经完成的 random holdout、scenario-group CV、contingency-group CV 和 top-k ranking 组织成最终论文的证据链。

核心原则：

- random holdout 用于快速比较模型；
- scenario-group CV 用于测试未见 operating scenario；
- contingency-group CV 用于测试未见 contingency；
- top-k ranking 用于模拟运行员只精算前 $k$ 个风险候选的场景。

In [19]:

# Cross-validation evidence synthesis table for teaching discussion
cv_evidence_rows = []
for cv_type, sub in group_cv_synthesis.groupby('cv_type'):
    best_by_recall = sub.sort_values(['mean_recall','mean_pf_calls_saved','mean_auprc'], ascending=False).iloc[0]
    cv_evidence_rows.append({
        'cv_type': cv_type,
        'best_model_by_recall': best_by_recall['model'],
        'stage': best_by_recall['stage'],
        'mean_recall': best_by_recall['mean_recall'],
        'mean_fnr': best_by_recall['mean_fnr'],
        'mean_pf_calls_saved': best_by_recall['mean_pf_calls_saved'],
        'evidence_file': 'week8_group_cv_synthesis.csv',
    })
cv_evidence = pd.DataFrame(cv_evidence_rows)
cv_evidence.to_csv(OUT_DIR / 'week8_cv_evidence_synthesis.csv', index=False)
cv_evidence


,cv_type,best_model_by_recall,stage,mean_recall,mean_fnr,mean_pf_calls_saved,evidence_file
0,contingency_group,EngineeringRule,Week 5 ML baseline,1.0,0.0,0.309524,week8_group_cv_synthesis.csv
1,scenario_group,EngineeringRule,Week 5 ML baseline,1.0,0.0,0.323232,week8_group_cv_synthesis.csv


## 19. Build final reproducibility package manifest

本节列出最终学生应提交的包结构。压缩包在 Notebook 外由课程脚本生成；Notebook 负责证明所有核心文件已经存在、非空、可追溯。

In [20]:

final_package_items = pd.DataFrame([
    {'category': 'paper', 'item': 'main.tex', 'path': repo_relative(PAPER_DIR / 'main.tex')},
    {'category': 'paper', 'item': 'paper tables', 'path': repo_relative(TABLE_DIR)},
    {'category': 'paper', 'item': 'paper figures', 'path': repo_relative(FIG_DIR)},
    {'category': 'presentation', 'item': 'final presentation outline', 'path': repo_relative(PRES_DIR / 'final_presentation_outline.md')},
    {'category': 'audit', 'item': 'artifact manifest', 'path': repo_relative(OUT_DIR / 'week8_artifact_manifest.csv')},
    {'category': 'audit', 'item': 'validation summary', 'path': repo_relative(OUT_DIR / 'week8_validation_summary.csv')},
    {'category': 'audit', 'item': 'claim evidence table', 'path': repo_relative(OUT_DIR / 'week8_claim_evidence_table.csv')},
    {'category': 'audit', 'item': 'reproducibility checklist', 'path': repo_relative(OUT_DIR / 'week8_reproducibility_checklist.csv')},
])
for p in final_package_items['path']:
    pp = PROJECT_ROOT / p
    assert pp.exists(), p
    if pp.is_file():
        assert pp.stat().st_size > 0, p
final_package_items.to_csv(OUT_DIR / 'week8_final_package_manifest.csv', index=False)
final_package_items


,category,item,path
0,paper,main.tex,08_week08_paper_reproducibility/outputs/paper/...
1,paper,paper tables,08_week08_paper_reproducibility/outputs/paper/...
2,paper,paper figures,08_week08_paper_reproducibility/outputs/paper/...
3,presentation,final presentation outline,08_week08_paper_reproducibility/outputs/presen...
4,audit,artifact manifest,08_week08_paper_reproducibility/outputs/week8_...
5,audit,validation summary,08_week08_paper_reproducibility/outputs/week8_...
6,audit,claim evidence table,08_week08_paper_reproducibility/outputs/week8_...
7,audit,reproducibility checklist,08_week08_paper_reproducibility/outputs/week8_...


## 20. Supplemental final checks

这些检查覆盖本周新增的 cross-validation synthesis 和最终包 manifest。

In [21]:

supplemental_checks = []
for name, path in {
    'cv_evidence_synthesis': OUT_DIR / 'week8_cv_evidence_synthesis.csv',
    'final_package_manifest': OUT_DIR / 'week8_final_package_manifest.csv',
}.items():
    supplemental_checks.append({'check': name, 'passed': path.exists() and path.stat().st_size > 0, 'detail': repo_relative(path)})

cv_ev = pd.read_csv(OUT_DIR / 'week8_cv_evidence_synthesis.csv')
supplemental_checks.append({'check': 'cv_evidence_metrics_bounded', 'passed': bool(cv_ev[['mean_recall','mean_fnr','mean_pf_calls_saved']].apply(lambda s: s.between(0,1).all()).all()), 'detail': 'CV metrics are in [0,1]'})

pkg_manifest = pd.read_csv(OUT_DIR / 'week8_final_package_manifest.csv')
supplemental_checks.append({'check': 'final_package_manifest_nonempty', 'passed': len(pkg_manifest) >= 8, 'detail': f'{len(pkg_manifest)} package items listed'})

supplemental = pd.DataFrame(supplemental_checks)
supplemental.to_csv(OUT_DIR / 'week8_supplemental_validation_summary.csv', index=False)
assert supplemental['passed'].all(), supplemental
supplemental


,check,passed,detail
0,cv_evidence_synthesis,True,08_week08_paper_reproducibility/outputs/week8_...
1,final_package_manifest,True,08_week08_paper_reproducibility/outputs/week8_...
2,cv_evidence_metrics_bounded,True,"CV metrics are in [0,1]"
3,final_package_manifest_nonempty,True,8 package items listed


## 18. Student tasks

1. 在 `paper/main.tex` 中补充 Introduction 和 Related Work；
2. 从 `paper/figures/` 选择 3--5 张最有说服力的图；
3. 修改 `week8_claim_evidence_table.csv`，确保每个 claim 都有 evidence；
4. 用 `presentation/final_presentation_outline.md` 制作 10--15 页答辩 PPT；
5. 在 limitation 部分明确说明 teaching dataset、小样本、islanding 尚未完整建模等限制。